In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Iris 데이터셋 준비
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

# 특성값의 크기를 맞춘 뒤 PyTorch Tensor로 변환
scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print(f"학습 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")
print(f"특성: {iris.feature_names}")
print(f"품종: {iris.target_names.tolist()}")


# 2. 신경망 모델 정의 (nn.Module 상속)
class MultiClassNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiClassNet, self).__init__()
        # 계층 정의
        self.fc1 = nn.Linear(input_size, hidden_size)  # 입력층 -> 은닉층
        self.sigmoid = nn.Sigmoid()                    # 활성화 함수
        self.fc2 = nn.Linear(hidden_size, output_size) # 은닉층 -> 출력층

    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)  # Softmax는 nn.CrossEntropyLoss 내부에서 처리되므로 생략
        return out

# 3. 모델, 손실 함수, 옵티마이저 생성
input_size = 4  # 꽃받침/꽃잎의 길이와 너비
hidden_size = 8
output_size = 3
learning_rate = 0.5

# 재현성을 위한 시드 고정
torch.manual_seed(42)

model = MultiClassNet(input_size, hidden_size, output_size)

# 손실 함수: Softmax + Cross-Entropy가 결합된 형태
criterion = nn.CrossEntropyLoss()

# 최적화 알고리즘: 경사하강법(SGD)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


# 4. 모델 학습 루프 (Training Loop)
print("=== PyTorch 학습 시작 ===")
epochs = 3000

for epoch in range(epochs):
    # ① 순전파 (Forward)
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    # ② 역전파 (Backward) 및 가중치 업데이트
    optimizer.zero_grad()  # 이전 스텝의 기울기(Gradient) 초기화
    loss.backward()        # 자동 미분을 통해 역전파 수행 (autograd)
    optimizer.step()       # 경사하강법으로 가중치 업데이트

    # 출력
    if (epoch + 1) % 500 == 0:
        # 가장 높은 확률 값을 가진 클래스 인덱스 추출
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y_train).float().mean() * 100
        print(f"Epoch {epoch + 1:4d} | Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.1f}%")


# 5. 테스트 데이터 평가 및 예측 결과 확인
print("\n=== Iris 테스트 결과 ===")
model.eval() # 평가 모드 전환

with torch.no_grad(): # 테스트 단계에서는 기울기 계산 불필요
    logits = model(X_test)
    # 실제 확률 분포를 보고 싶다면 torch.softmax 적용
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(probabilities, dim=1)

test_accuracy = (predictions == y_test).float().mean() * 100
print(f"테스트 정확도: {test_accuracy.item():.1f}% ({(predictions == y_test).sum().item()}/{len(y_test)})")

print("\n처음 10개 테스트 샘플의 예측:")
for i, (prob, pred, actual) in enumerate(zip(probabilities[:10], predictions[:10], y_test[:10])):
    prob_list = [round(p, 3) for p in prob.tolist()]
    print(f"샘플 {i+1:2d} | 실제: {iris.target_names[actual.item()]:10s} | "
          f"예측: {iris.target_names[pred.item()]:10s} | 확률: {prob_list}")